# EXOME BURDEN LOGISTIC REGRESSION 
This script runs a logistic regression for unrelated individuals using the gene-set rare polygenic risk score, the exome burden polygenic risk score, principal components, age, and sex to identify gene sets that harbor rare variants significantly associated with acute myeloid leukaemia.

In [ ]:
library(data.table)
library(tidyverse)
library(readr)
library(logistf)

In [ ]:
# Ensure a clean directory for the logistic regression loop 
system("rm -rf burden_polygenic_score")
system("mkdir -p burden_polygenic_score")
system("rm -rf logistic_summary")
system("mkdir -p logistic_summary")

In [ ]:
# Copy gene-set burden score files to directory 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/*PRS.tsv burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/rna_seq_differential_expression/*PRS.tsv burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/rna_seq_cox_regression/*PRS.tsv burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/single_cell_differential_expression/*PRS.tsv burden_polygenic_score")

In [ ]:
# List of file present in the burden polygenic score folder 
files <- list.files(path="burden_polygenic_score")

In [ ]:
# Load files necessary for logistic regeression 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/exome/* burden_polygenic_score")
system("gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/pcs/EUR_flashpca.pcs burden_polygenic_score", intern=TRUE)

In [ ]:
# Load phenotype file 
pheno <- fread("burden_polygenic_score/EUR_phenotype_file.txt")
 # Load exome burden file 
exome <- fread("burden_polygenic_score/exome-burden-PRS.tsv")
 # Load genomic metric file 
metrics <- fread("burden_polygenic_score/genomic_metrics.tsv")
# Format exome burden file 
exome <- rename(exome, 
                "person_id" = "IID", 
                "exome_burden_prs" = "EXOME_SCORE1_SUM") %>% 
         select(person_id, exome_burden_prs)
# Format genomic metric file 
metrics <- select(rename(metrics, "person_id" = "research_id"), person_id, sample_source)
 # Load relationship file 
eur_flashpca <- fread("burden_polygenic_score/EUR_flashpca.pcs") %>% 
    dplyr::select(IID, group) %>% 
    rename('person_id' = 'IID') %>% 
    filter(group == "Unrelated_Inliers")

In [ ]:
# =============================================================================================================
# Logistic regression of gene-sets for exome burden polygenic risk score 
# =============================================================================================================

# Loop for running logistic regression for each gene-set 
for (file in files){
    print(paste0("Processing file ", gsub("-PRS.tsv", "", file) , " ...."))
   
    # Load gene-set burden score 
    score <- fread(paste0("burden_polygenic_score/", file)) 
    # Rename column present in score funciotn 
    score <- rename(score, 
                "person_id" = "IID", 
                "burden_prs" = "TOTAL_SCORE1_SUM") %>% 
         select(person_id, burden_prs)
    # List of data frames
    df_list <- list(pheno, score, exome, metrics, eur_flashpca)

    # Merge all data frames 
    merged_df <- Reduce(function(x, y) inner_join(x, y, by = "person_id"), df_list)

    # No rare variant present in gene set, skip gene set 
    if (max(merged_df$burden_prs) == 0){
        print(paste0("Skipping file ", gsub("-PRS.tsv", "", file) , " ...."))
        next 
    }
    # Scale remaining unscale numeric columns 
    merged_df[, c("burden_prs", "exome_burden_prs")] <-
                        as.data.frame(scale(merged_df[, c("burden_prs", "exome_burden_prs")]))

    # Run logistic regression 
    logistic <- 
        glm(aml_phenotype ~ burden_prs + exome_burden_prs + sex + age + 
            PC1 + PC2 + PC3 + PC4 + PC5 + 
            PC6 + PC7 + PC8 + PC9 + PC10, 
            data = merged_df, 
            family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
                        
    flush.console()  # Forces immediate output
    Sys.sleep(0.5)   # Example delay
                        
    # Save summary
    write_tsv(as.data.frame(logistic_coef),
              paste0("logistic_summary/", gsub("-PRS.tsv", "", file), "-logistic-summary.tsv"))
                        
}

In [ ]:
# Copy logistic summary to workspace bucket  
system("gsutil cp logistic_summary/* gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_unrelated_logistic_summary/")